This is a conversational RAG application using LangChain and openAI LLM model. It can perform the conversation based on LLM model, and other additional project data which we input into the RAG architecture. Furthermore it can maintain a cha t history to memorize previous chats and repsonse accordinly 

In [4]:
#Install required packages
#langchain framework, chains gents etc
!pip install langchain -qU 
#to use openAI specific integrations like LLM and embedding model
!pip install langchain-openai -qU
#To use chroma vectordatabase
!pip install langchain-chroma -qU
#To use other thrid party product integration 
!pip install langchain_community -qU 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
#call the API key as an environment variable
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() /'.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)


Note: you may need to restart the kernel to use updated packages.
Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\LangChain\.env
File exists: True
Loaded: True



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
#initialize the OpenAI LLM
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-3.5-turbo",temperature=0,openai_api_key=api_key)

In [11]:
#initialize the embedding model
#we use a openAI embedding model
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model="text-embedding-3-small")

In [13]:
#load pdf docuemnt
from langchain_community.document_loaders import PyPDFLoader
#load the PDF Document
loader=PyPDFLoader("/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/IPC_2025_Paper_final_version.pdf")
docs=loader.load()

In [ ]:
#checking the loaded document
len(docs) #gives no of pages

2

In [15]:
#first page data, contnent and meta data
docs[0]

Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-05-03T19:35:44+05:30', 'title': 'Paper Title (use style: paper title)', 'author': 'IEEE', 'moddate': '2025-05-03T19:35:44+05:30', 'source': '/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/IPC_2025_Paper_final_version.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Polarization-Aware Modular IR Apertures for \nInterference Mitigation in Multi-User Indoor \nNetworks \nS. Gunathilake1, A. Nirmalathas2, K. Herath2 and M. Premaratne1 \n1. Department of Electrical and Computer Systems Engineering, Monash University, Clayton, VIC 3800, Australia \n2. Department of Electrical and Electronic Engineering, The University of Melbourne, VIC 3010, Australia \nMalin.premaratne@monash.edu\nAbstract—A polarization -aware method improves modular \nclustered IR optical apertures to mitigate inter -user interference \nin multi -user near -field indoor n

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#intialize the text splitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
#split the documents into chunks
splits=text_splitter.split_documents(docs)

In [18]:
len(splits)

26

In [19]:
splits[0]

Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-05-03T19:35:44+05:30', 'title': 'Paper Title (use style: paper title)', 'author': 'IEEE', 'moddate': '2025-05-03T19:35:44+05:30', 'source': '/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/IPC_2025_Paper_final_version.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Polarization-Aware Modular IR Apertures for \nInterference Mitigation in Multi-User Indoor \nNetworks \nS. Gunathilake1, A. Nirmalathas2, K. Herath2 and M. Premaratne1 \n1. Department of Electrical and Computer Systems Engineering, Monash University, Clayton, VIC 3800, Australia \n2. Department of Electrical and Electronic Engineering, The University of Melbourne, VIC 3010, Australia')

In [30]:
#Create the vector store and the retriever
from langchain_chroma import Chroma

#Create a vector store from the document chunks
vectorStore=Chroma.from_documents(documents=splits,embedding=embedding_model)

#Crate the retirever
retriever=vectorStore.as_retriever()

In [23]:
from langchain_core.prompts import ChatPromptTemplate

#define system prompt, for this system prompt we get the cotext from the vectorstore matching chunks, we get this through the retirever
system_prompt=("You are an intelligent chatbot. Use the following context to answer the question. If you dont know the answer, just say that {context}")

#create the prompt template
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
    )

In [ ]:
prompt # gieve the prompt detaisl, here the context is coming from retirever,and the input is the question from the user

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are an intelligent chatbot. Use the following context to answer the question. If you dont know the answer, just say that {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [ ]:
#create the RAG chain
#We create this RAG chain using Question-Answering (QA---LLM+Prompt) chain and the retrever(to get the context from documents) 
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_document_

#first create the Question-answering chain
qa_chain=create_stuff_documents_chain(llm,prompt)

#create the RAG chain
rag_chain=create_retrieval_chain(retriever,qa_chain)

ModuleNotFoundError: No module named 'langchain.chains'

In [31]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG chain using  retirever+QA chain  where QA chain=llm+prompt, here runnablrPassthrough is used to pass the orginal question from the user to the chain
rag_chain = (
    {"context": retriever, "input": RunnablePassthrough()} # retirever + input
    | prompt #make ready the human input,retirever and system prompt to feed to llm
    | llm #actual model
    | StrOutputParser() #llm returns and message, parser extract just the string instrad of that object
)

In [ ]:
#invoke RAG chain with example questions
response=rag_chain.invoke("what are polarization aware modular IR clusters")
print(response)

Polarization-aware modular IR clusters are a type of optical aperture framework designed to reduce inter-user interference in multi-user near-field indoor networks. These clusters assign orthogonal polarization with aligned receivers to enhance the Signal-to-Interference-plus-Noise Ratio (SINR) and improve system reliability. This approach results in a 77% reduction in average Bit Error Rate (BER), indicating increased performance and reliability in communication systems.


In [ ]:
response2=rag_chain.invoke("what is RAG architecture")
print(response2) #Since given pdf article doesnt include those detials, it doesnt know, it, and says it doesnt know.

I'm sorry, but I don't have specific information about the RAG architecture in the provided context. If you have more details or context about it, I can try to provide a better answer.


In [ ]:
response3=rag_chain.invoke("how we reduce the interfernence in them") #seems like doesnt maintin abeteer chat history yet to answer
print(response3)

To reduce interference in the system, the proposed method focuses on alignment at the receiver to effectively mitigate inter-user interference. This alignment strategy helps improve the Signal-to-Interference-plus-Noise Ratio (SINR) and reduces the Bit Error Rate (BER) performance. The simulation results show substantial enhancements in performance, offering a simple yet effective extension to the prior architecture.


Adding chat hisotry

In [44]:
#Correct imports for langchain v1.x
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

contextualize_system_prompt = (
    "Given the chat history and the latest user question, "
    "reformulate the question to be standalone if needed. "
    "Otherwise return it as is. Do NOT answer the question."
)

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

#history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

#QA prompt
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an intelligent chatbot. Use the context to answer. "
               "If you don't know, say you don't know.\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

#Full RAG chain
document_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

In [ ]:
#Manage cht session history
#base on the session id we prepare a discionary to store the chat history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#initialize the store dict for session histories
store={}# this dictonary can store ina database or jason fiel later

#function to get the session history for a given session ID
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

#create the conversational RAG chain with session history
conversational_rag_chain=RunnableWithMessageHistory(rag_chain,get_session_history,input_messages_key="input",history_messages_key="chat_history",output_messages_key="answer",)


In [47]:
#invoke the conversational_rag_chain with example questions
response=conversational_rag_chain.invoke({"input":"what are IR radiative clusters"},config={"configurable":{"session_id":"101"}},)
response["answer"]

'IR radiative element clusters are groups of infrared (IR) elements arranged in a specific pattern within an optical aperture architecture. These clusters are designed to improve transmission efficiency in optical wireless networks by focusing and directing the IR signals effectively.'

In [ ]:
response=conversational_rag_chain.invoke({"input":"how polarization join here"},config={"configurable":{"session_id":"101"}},)
response["answer"] #so now answer has given how polarization can manage for previously mentioned cluster IR arrays

"In the context provided, polarization is being utilized strategically within the IR optical aperture framework to enhance the system's performance. By assigning orthogonal polarizations to radiative sub-clusters and aligning receivers accordingly, the system can improve the signal-to-interference-plus-noise ratio (SINR) and overall reliability. This polarization-aware approach helps mitigate inter-user interference in multi-user near-field indoor networks, leading to a significant reduction in bit error rate (BER) and improved system reliability."

In [49]:
response=conversational_rag_chain.invoke({"input":"how polarization join here"},config={"configurable":{"session_id":"103"}},)
response["answer"] #so now answer has given how polarization can manage for previously mentioned cluster IR arrays

'In this context, polarization is being used to refer to the orientation of the electric field in an electromagnetic wave. By incorporating polarization vectors that match at both the optical aperture and receiver end, the design aims to leverage selective interference mitigation through polarization orthogonality. This means that by aligning the polarization vectors of the transmitted and received signals, interference can be reduced without changing the physical configuration of the aperture.'